# SQL Date and String Functions — Runnable Examples

This notebook uses Python's built-in SQLite database for practical string/date examples. SQLite date functions differ from PostgreSQL, MySQL, and SQL Server; SQL-dialect examples are shown in markdown where syntax is vendor-specific.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.execute("""CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    full_name TEXT,
    email TEXT,
    city TEXT
)""")
customers = [
    (1, "  Asha Sharma ", "ASHA@example.com", "Indore"),
    (2, "Ravi Patel", "ravi@example.com", "Bhopal"),
    (3, " neha singh", "NEHA@example.org", "Indore "),
    (4, "Mohan Das", None, None),
]
conn.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", customers)
pd.read_sql_query("SELECT * FROM customers ORDER BY customer_id", conn)


,customer_id,full_name,email,city
0,1,Asha Sharma,ASHA@example.com,Indore
1,2,Ravi Patel,ravi@example.com,Bhopal
2,3,neha singh,NEHA@example.org,Indore
3,4,Mohan Das,None,None


## 1. Case conversion and trimming

In [2]:
pd.read_sql_query("""
SELECT customer_id,
       UPPER(TRIM(city)) AS city_upper,
       LOWER(TRIM(email)) AS normalized_email,
       TRIM(full_name) AS clean_name
FROM customers
ORDER BY customer_id
""", conn)


,customer_id,city_upper,normalized_email,clean_name
0,1,INDORE,asha@example.com,Asha Sharma
1,2,BHOPAL,ravi@example.com,Ravi Patel
2,3,INDORE,neha@example.org,neha singh
3,4,None,None,Mohan Das


## 2. Length, substring, and replacement

In [3]:
pd.read_sql_query("""
SELECT full_name,
       LENGTH(TRIM(full_name)) AS name_length,
       SUBSTR(TRIM(full_name), 1, 5) AS first_five,
       REPLACE(TRIM(city), ' ', '_') AS city_slug
FROM customers
ORDER BY customer_id
""", conn)


,full_name,name_length,first_five,city_slug
0,Asha Sharma,11,Asha,Indore
1,Ravi Patel,10,Ravi,Bhopal
2,neha singh,10,neha,Indore
3,Mohan Das,9,Mohan,None


## 3. Concatenation and NULL handling

SQLite's `||` operator propagates NULL. `COALESCE` provides a fallback.

In [4]:
pd.read_sql_query("""
SELECT customer_id,
       TRIM(full_name) || ' <' || COALESCE(email, 'no-email') || '>' AS contact_label,
       COALESCE(city, 'Unknown') AS city_or_unknown
FROM customers
ORDER BY customer_id
""", conn)


,customer_id,contact_label,city_or_unknown
0,1,Asha Sharma <ASHA@example.com>,Indore
1,2,Ravi Patel <ravi@example.com>,Bhopal
2,3,neha singh <NEHA@example.org>,Indore
3,4,Mohan Das <no-email>,Unknown


## 4. Find the email username

`INSTR` returns the position of `@` in SQLite. This is a simple example, not full email validation.

In [5]:
pd.read_sql_query("""
SELECT email,
       CASE WHEN email IS NOT NULL AND INSTR(email, '@') > 0
            THEN SUBSTR(email, 1, INSTR(email, '@') - 1)
            ELSE NULL
       END AS email_user
FROM customers
ORDER BY customer_id
""", conn)


,email,email_user
0,ASHA@example.com,ASHA
1,ravi@example.com,ravi
2,NEHA@example.org,NEHA
3,None,None


## 5. Filter a case-normalized domain

In [6]:
pd.read_sql_query("""
SELECT customer_id, email
FROM customers
WHERE LOWER(email) LIKE '%@example.com'
ORDER BY customer_id
""", conn)


,customer_id,email
0,1,ASHA@example.com
1,2,ravi@example.com


## 6. Date table and extracting date parts

In [7]:
conn.execute("""CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    order_timestamp TEXT,
    shipped_date TEXT,
    amount REAL
)""")
orders = [
    (101, "2026-01-03 09:15:00", "2026-01-08", 120.0),
    (102, "2026-01-15 17:40:00", "2026-01-25", 250.0),
    (103, "2026-02-01 10:00:00", None, 75.0),
    (104, "2026-02-20 12:30:00", "2026-03-04", 310.0),
]
conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
pd.read_sql_query("""
SELECT order_id, order_timestamp,
       STRFTIME('%Y', order_timestamp) AS order_year,
       STRFTIME('%m', order_timestamp) AS order_month,
       DATE(order_timestamp) AS order_date
FROM orders
ORDER BY order_timestamp
""", conn)


,order_id,order_timestamp,order_year,order_month,order_date
0,101,2026-01-03 09:15:00,2026,01,2026-01-03
1,102,2026-01-15 17:40:00,2026,01,2026-01-15
2,103,2026-02-01 10:00:00,2026,02,2026-02-01
3,104,2026-02-20 12:30:00,2026,02,2026-02-20


## 7. Half-open range for January 2026

In [8]:
pd.read_sql_query("""
SELECT order_id, order_timestamp, amount
FROM orders
WHERE order_timestamp >= '2026-01-01'
  AND order_timestamp <  '2026-02-01'
ORDER BY order_timestamp
""", conn)


,order_id,order_timestamp,amount
0,101,2026-01-03 09:15:00,120.0
1,102,2026-01-15 17:40:00,250.0


## 8. Days between order and shipment

SQLite's `JULIANDAY` can calculate a day difference for ISO date strings.

In [9]:
pd.read_sql_query("""
SELECT order_id, DATE(order_timestamp) AS order_date, shipped_date,
       CASE WHEN shipped_date IS NOT NULL
            THEN CAST(JULIANDAY(shipped_date) - JULIANDAY(DATE(order_timestamp)) AS INTEGER)
            ELSE NULL
       END AS days_to_ship
FROM orders
ORDER BY order_id
""", conn)


,order_id,order_date,shipped_date,days_to_ship
0,101,2026-01-03,2026-01-08,5.0
1,102,2026-01-15,2026-01-25,10.0
2,103,2026-02-01,None,NaN
3,104,2026-02-20,2026-03-04,12.0


## 9. Monthly order counts

In [10]:
pd.read_sql_query("""
SELECT STRFTIME('%Y-%m', order_timestamp) AS order_month,
       COUNT(*) AS order_count,
       ROUND(SUM(amount), 2) AS monthly_amount
FROM orders
GROUP BY STRFTIME('%Y-%m', order_timestamp)
ORDER BY order_month
""", conn)


,order_month,order_count,monthly_amount
0,2026-01,2,370.0
1,2026-02,2,385.0


## 10. Orders shipped more than 10 days after ordering

In [11]:
pd.read_sql_query("""
SELECT order_id,
       CAST(JULIANDAY(shipped_date) - JULIANDAY(DATE(order_timestamp)) AS INTEGER) AS days_to_ship
FROM orders
WHERE shipped_date IS NOT NULL
  AND JULIANDAY(shipped_date) - JULIANDAY(DATE(order_timestamp)) > 10
ORDER BY order_id
""", conn)


,order_id,days_to_ship
0,104,12


## Dialect notes

- PostgreSQL date parts: `EXTRACT(YEAR FROM order_date)`; format with `TO_CHAR`; truncate with `DATE_TRUNC`.
- MySQL formatting: `DATE_FORMAT(order_date, '%Y-%m')`; arithmetic often uses `DATE_ADD` / `DATE_SUB`.
- SQL Server commonly uses `DATEPART`, `DATEADD`, `DATEDIFF`, and `LEN`.
- SQLite uses `STRFTIME`, `DATE`, `DATETIME`, `JULIANDAY`, and `SUBSTR` for many similar tasks.

Always check the SQL dialect and the timezone/format of source data.